In [3]:
from transformers import CLIPProcessor, CLIPModel
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import Imagenette
from sklearn.metrics import accuracy_score
import numpy as np
import torchvision
from tqdm import tqdm

/home/shkaf2m/Desktop/ml-isp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [ ]:
def custom_collate_function(batch):
  images = [item[0] for item in batch]
  labels = [item[1] for item in batch]
  labels = torch.tensor(labels, dtype=torch.long)
  return images, labels

In [ ]:
val_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
])

val_dataset = Imagenette(root='./data', split='val', download=True, transform = val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=7, collate_fn = custom_collate_function)

In [9]:
def ValidateModel(model, device, data_loader, all_classes):
  model.eval()
  true_labels = torch.tensor([]).to(DEVICE)
  predicted_labels = torch.tensor([]).to(DEVICE)

  with torch.no_grad():
    for images, labels in tqdm(data_loader):
      labels = labels.to(DEVICE)
      inputs = processor(text = all_classes, images = images, return_tensors = "pt", padding = True).to(DEVICE)
      
      outputs = model(**inputs)
      predicted = outputs.logits_per_image.argmax(dim=-1)
      true_labels = torch.cat((true_labels, labels), 0)
      predicted_labels = torch.cat((predicted_labels, predicted), 0)

  return true_labels, predicted_labels

In [10]:
import os
from sklearn.metrics import f1_score
os.environ["TOKENIZERS_PARALLELISM"] = "false"

all_classes = [label[0] for label in val_dataset.classes]

model = model.to(DEVICE)
model.eval()
true_data, predicted_data = ValidateModel(model, device = DEVICE, data_loader = val_loader, all_classes = all_classes)
f1_res = f1_score(true_data.cpu().numpy(), predicted_data.cpu().numpy(), average = 'macro')

print("F1 Score: ", f1_res)

100%|██████████| 123/123 [00:36<00:00,  3.38it/s]

F1 Score:  0.9876700606902598
